<a href="https://colab.research.google.com/github/An-n-nie/beginning-bioinformatics/blob/main/Module5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intsalling libraries and Checking Installation

In [ ]:
#install Biopython, MAFT
!pip install biopython
!apt-get update -qq
!apt-get install -y mafft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.6 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-lato libauthen-sasl-perl libclone-perl libdata-dump-perl
  libencode-locale-perl libfile-listing-perl libfont-afm-perl
  libhtml-form-perl libhtml-format-perl libhtml-parser-perl
  libhtml-tagset-perl libhtml-tree-perl libhttp-cookies-perl
  libhttp-daemon-perl libhttp-date-perl libhttp-message-perl
  libhttp-negotiate-perl libio-html-perl libio-socket-ssl-perl
  liblwp-mediatypes-perl liblwp-protocol-https-perl libmailtools-perl
  libnet-http-perl libnet-smtp-ssl-perl libnet-ssleay-perl libruby libruby3.2
  libtry-tiny-perl liburi-perl libwww-perl libwww-robotrules-per

In [ ]:
!mafft --version

v7.505 (2022/Apr/10)


In [ ]:
import Bio
print(Bio.__version__)

1.88


# Alignment w MAFT

In [ ]:
#MAFT alignment code

!mafft --auto P450.fasta > P450_MAFFT.fasta

nthread = 0
nthreadpair = 0
nthreadtb = 0
ppenalty_ex = 0
stacksize: 8192 kb
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00



Making a distance matrix ..

There are 8 ambiguous characters.
  101 / 129
done.

Constructing a UPGMA tree (efffree=0) ... 
  120 / 129
done.

Progressive alignment 1/2... 
STEP   114 / 128  f
Reallocating..done. *alloclen = 4930
STEP   128 / 128  f
done.

Making a distance matrix from msa.. 
  100 / 129
done.

Constructing a UPGMA tree (efffree=1) ... 
  120 / 129
done.

Progressive alignment 2/2... 
STEP   119 / 128  f
Reallocating..done. *alloclen = 4275
STEP   128 / 128  f
done.

disttbfast (aa) Version 7.505
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)

rescale = 1
dndpre (aa) Version 7.505
alg=X, model=BLOSUM62, 1.53, +0.12, -0.00, noshift, amax=0.0
0 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 2
sueff_global = 0.100000
nadd = 2
re

MAFT vs MUSCLE alignments

In [ ]:
from Bio import AlignIO

muscle = AlignIO.read("p450maxcc.afa", "fasta") #best alignment file from MUSCLE done on Bridges 2
mafft = AlignIO.read("P450_MAFFT.fasta", "fasta") #MAFT alignment we just made

#Check if these alignments share same length - they don't
print("MUSCLE length:", muscle.get_alignment_length())
print("MAFFT length:", mafft.get_alignment_length())

MUSCLE length: 4132
MAFFT length: 2871


In [ ]:
#Check if the alignments really came from same source

muscle_id = {record.id for record in muscle}
mafft_id = {record.id for record in mafft}

print(muscle_id == mafft_id)

True


In [17]:
#Checking amount of columns that has gap ( - ) in both alignments - amounts varied for both
def gap_columns(alignment):
    count = 0

    for column in range(alignment.get_alignment_length()):
        column_chars = alignment[:, column]

        if "-" in column_chars:
            count += 1

    return count

print("MUSCLE columns with >=1 gaps:", gap_columns(muscle))

print("MAFFT columns with >=1 gaps:", gap_columns(mafft))

MUSCLE columns with >=1 gaps: 4123
MAFFT columns with >=1 gaps: 2863


In [ ]:
#Check how many gaps each alignment introduced - MUSCLE created more gaps
def total_gaps(alignment):
    count = 0
    for record in alignment:
        count += record.seq.count('-')
    return count

print("MUSCLE total gaps:", total_gaps(muscle))
print("MAFFT total gaps:", total_gaps(mafft))

MUSCLE total gaps: 465711
MAFFT total gaps: 303042


In [19]:
#Compare gap placement between MUSCLE and MAFFT

from collections import defaultdict

def get_gap_profile(sequence):

    profile = defaultdict(int)
    residue_count = 0

    for char in str(sequence):

        if char == "-":
            profile[residue_count] += 1

        else:
            residue_count += 1

    return dict(profile)


# Match sequences by ID
muscle_dict = {record.id: record for record in muscle}
mafft_dict = {record.id: record for record in mafft}

common_ids = sorted(set(muscle_dict) & set(mafft_dict))


# Overall results
exact_same_profiles = 0
different_profiles = 0

total_shared_gap_positions = 0
total_gap_positions = 0


for seq_id in common_ids:

    muscle_profile = get_gap_profile(muscle_dict[seq_id].seq)
    mafft_profile = get_gap_profile(mafft_dict[seq_id].seq)

    # Did the two methods put gaps at exactly the same positions
    # AND with exactly the same gap lengths?
    if muscle_profile == mafft_profile:
        exact_same_profiles += 1
    else:
        different_profiles += 1

    # Compare gap locations, ignoring differences in gap length
    muscle_positions = set(muscle_profile.keys())
    mafft_positions = set(mafft_profile.keys())

    shared_positions = muscle_positions & mafft_positions
    all_positions = muscle_positions | mafft_positions

    total_shared_gap_positions += len(shared_positions)
    total_gap_positions += len(all_positions)


# Calculate percentage of sequences with identical gap profiles
profile_percent = (exact_same_profiles / len(common_ids)) * 100

# Calculate overall similarity of gap locations
if total_gap_positions > 0:
    gap_location_similarity = (
        total_shared_gap_positions / total_gap_positions
    ) * 100
else:
    gap_location_similarity = 100


#Final results

print(f"Sequences compared: {len(common_ids)}")

print("\nExact gap profiles:")
print(f"Same gap placement + length: {exact_same_profiles}")
print(f"Different: {different_profiles}")
print(f"Percentage identical: {profile_percent:.2f}%")

print("\nGap-location similarity:")
print(f"Shared gap positions: {total_shared_gap_positions:,}")
print(f"All unique gap positions: {total_gap_positions:,}")
print(f"Gap-location similarity: {gap_location_similarity:.2f}%")

Sequences compared: 129

Exact gap profiles:
Same gap placement + length: 0
Different: 129
Percentage identical: 0.00%

Gap-location similarity:
Shared gap positions: 2,518
All unique gap positions: 17,642
Gap-location similarity: 14.27%


### Given the same files, MAFT and MUSCLE did not produce the same alignments. Their alignments had different lengths with MUSCLE being longer and also introduced more gaps than MAFT. Of the gaps introduced by both alignments, none of them share the same pattern. Of the distinct gap positions, only about 14% of them is shared by the two alignments.